# SleepFM-Clinical: Evaluation Notebook

**SleepFM** is a multimodal sleep foundation model (Nature Medicine, 2026) that predicts 130+ diseases from overnight polysomnography (PSG) data.

Key results from the paper:
- Mortality prediction: C-Index 0.84
- Dementia prediction: C-Index 0.85
- 5-class sleep staging from learned embeddings

This notebook runs the full inference pipeline:
1. **Setup** — clone repo, install dependencies, verify GPU
2. **Preprocessing** — convert EDF → HDF5 (synthetic demo data)
3. **Embedding Generation** — extract multimodal embeddings with pretrained SetTransformer
4. **Sleep Staging** — 5-class predictions (Wake, Stage 1-3, REM) with confusion matrix & hypnogram
5. **Disease Prediction** — risk scores for 1,065 conditions with visualization
6. **Advanced Visualizations** — t-SNE embedding clusters
7. **Real Data** — instructions for Stanford Sleep Dataset

**Runtime**: GPU (T4 free tier is sufficient for inference)

> All demo data in this notebook is synthetic. Real clinical data requires appropriate IRB approval and data use agreements.

---
## 1. Setup & Installation

In [ ]:
# Mount Google Drive for persistent storage (optional — needed for real data)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository
!git clone https://github.com/healthvai/sleepfm-clinical.git /content/sleepfm-clinical 2>/dev/null || echo "Already cloned"
%cd /content/sleepfm-clinical

In [ ]:
# Install dependencies
# Note: Colab already has PyTorch + CUDA pre-installed.
# pyedflib's C extension needs numpy ABI compatibility — downgrade numpy
# first, then force-reinstall pyedflib to get the right binary.
!pip install "numpy<2" --quiet
!pip install --force-reinstall --no-cache-dir pyedflib --quiet
!pip install h5py loguru einops mne \
    statsforecast statsmodels wandb seaborn pyyaml --quiet

# Verify GPU
import torch
assert torch.cuda.is_available(), "GPU not available! Go to Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
props = torch.cuda.get_device_properties(0)
gpu_mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
print(f"GPU Memory: {gpu_mem / 1e9:.1f} GB")
import numpy; print(f"NumPy: {numpy.__version__}")

In [ ]:
# Core imports
import torch
from torch import nn
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.manifold import TSNE
import os
import tqdm
import random
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from collections import Counter
import pandas as pd
import h5py
from torch.utils.data import Dataset, DataLoader

# Add repo paths
REPO_ROOT = "/content/sleepfm-clinical"
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "sleepfm"))

from preprocessing.preprocessing import EDFToHDF5Converter
from models.dataset import SetTransformerDataset, collate_fn
from models.models import SetTransformer, SleepEventLSTMClassifier, DiagnosisFinetuneFullLSTMCOXPHWithDemo
from utils import load_config, load_data, save_data, count_parameters

# Paths
DEMO_DATA = os.path.join(REPO_ROOT, "notebooks", "demo_data")
CHECKPOINTS = os.path.join(REPO_ROOT, "sleepfm", "checkpoints")
CONFIGS = os.path.join(REPO_ROOT, "sleepfm", "configs")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Demo data: {os.listdir(DEMO_DATA)}")

---
## 2. Preprocessing: EDF → HDF5

PSG recordings come as EDF (European Data Format) files. The preprocessing step:
- Reads raw EDF channels
- Resamples all channels to 128 Hz
- Applies lowpass Butterworth filter before downsampling
- Z-score normalizes each channel
- Saves to HDF5 with gzip compression

The channels are grouped into 4 modalities:
- **BAS** (Brain Activity/Sleep): EEG + EOG channels
- **RESP** (Respiratory): Chest/abdomen, SpO2, nasal airflow
- **EKG** (Cardiac): ECG leads
- **EMG** (Muscle): Chin, leg EMG channels

In [ ]:
# Convert synthetic demo EDF to HDF5
edf_path = os.path.join(DEMO_DATA, "demo_psg.edf")
hdf5_path = os.path.join(DEMO_DATA, "demo_psg.hdf5")

converter = EDFToHDF5Converter(
    root_dir="/dummy",
    target_dir="/dummy",
    resample_rate=128
)

converter.convert(edf_path, hdf5_path)
print(f"\nConverted: {edf_path} -> {hdf5_path}")

# Inspect the HDF5 structure
with h5py.File(hdf5_path, 'r') as f:
    print(f"\nHDF5 contents:")
    for key in f.keys():
        print(f"  {key}: shape={f[key].shape}, dtype={f[key].dtype}")

---
## 3. Embedding Generation

The pretrained **SetTransformer** (4.44M params) processes each modality independently:

```
Raw signal → 1D CNN Tokenizer → Spatial Attention Pooling → 
Positional Encoding → 6-layer Transformer → Temporal Attention Pooling
```

Produces two types of embeddings per modality:
- **5-second granular** embeddings (for sleep staging)
- **5-minute aggregated** embeddings (for disease prediction)

In [ ]:
# Load base model config
model_path = os.path.join(CHECKPOINTS, "model_base")
channel_groups_path = os.path.join(CONFIGS, "channel_groups.json")
config_path = os.path.join(model_path, "config.json")

config = load_config(config_path)
channel_groups = load_data(channel_groups_path)

# Extract model params
modality_types = config["modality_types"]
in_channels = config["in_channels"]
patch_size = config["patch_size"]
embed_dim = config["embed_dim"]
num_heads = config["num_heads"]
num_layers = config["num_layers"]
pooling_head = config["pooling_head"]
dropout = 0.0  # No dropout for inference

print(f"Model: {config['model']}")
print(f"Modalities: {modality_types}")
print(f"Architecture: {embed_dim}d, {num_heads} heads, {num_layers} layers")
print(f"Patch size: {patch_size} samples ({patch_size/128:.1f}s at 128Hz)")

In [ ]:
# Build and load pretrained SetTransformer
model_class = getattr(sys.modules[__name__], config['model'])
base_model = model_class(
    in_channels, patch_size, embed_dim, num_heads, num_layers,
    pooling_head=pooling_head, dropout=dropout
)

base_model = torch.nn.DataParallel(base_model)
base_model.to(device)

# Load pretrained weights
checkpoint = torch.load(os.path.join(model_path, "best.pt"), map_location=device, weights_only=False)
base_model.load_state_dict(checkpoint["state_dict"])
base_model.eval()

total_layers, total_params = count_parameters(base_model)
print(f"Pretrained SetTransformer loaded successfully!")
print(f"Parameters: {total_params / 1e6:.2f}M across {total_layers} layers")

In [ ]:
# Create dataset and dataloader
hdf5_paths = [os.path.join(DEMO_DATA, "demo_psg.hdf5")]
dataset = SetTransformerDataset(config, channel_groups, hdf5_paths=hdf5_paths, split="test")
dataloader = DataLoader(dataset, batch_size=16, num_workers=1, shuffle=False, collate_fn=collate_fn)

print(f"Dataset: {len(dataset)} samples, {len(dataloader)} batches")

In [ ]:
# Generate embeddings for all 4 modalities
output_emb = os.path.join(DEMO_DATA, "demo_emb")
output_5min_agg = os.path.join(DEMO_DATA, "demo_5min_agg_emb")
os.makedirs(output_emb, exist_ok=True)
os.makedirs(output_5min_agg, exist_ok=True)

with torch.no_grad():
    for batch in tqdm.tqdm(dataloader, desc="Generating embeddings"):
        batch_data, mask_list, file_paths, dset_names_list, chunk_starts = batch
        (bas, resp, ekg, emg) = batch_data
        (mask_bas, mask_resp, mask_ekg, mask_emg) = mask_list

        # Move to GPU
        bas = bas.to(device, dtype=torch.float)
        resp = resp.to(device, dtype=torch.float)
        ekg = ekg.to(device, dtype=torch.float)
        emg = emg.to(device, dtype=torch.float)
        mask_bas = mask_bas.to(device, dtype=torch.bool)
        mask_resp = mask_resp.to(device, dtype=torch.bool)
        mask_ekg = mask_ekg.to(device, dtype=torch.bool)
        mask_emg = mask_emg.to(device, dtype=torch.bool)

        # Forward pass for each modality
        embeddings = [
            base_model(bas, mask_bas),
            base_model(resp, mask_resp),
            base_model(ekg, mask_ekg),
            base_model(emg, mask_emg),
        ]

        # Save 5-minute aggregated embeddings
        embeddings_5min = [e[0].unsqueeze(1) for e in embeddings]
        for i in range(len(file_paths)):
            subject_id = os.path.basename(file_paths[i]).split('.')[0]
            chunk_start = chunk_starts[i]
            out_path = os.path.join(output_5min_agg, f"{subject_id}.hdf5")
            with h5py.File(out_path, 'a') as hf:
                for mod_idx, mod_type in enumerate(modality_types):
                    data = embeddings_5min[mod_idx][i].cpu().numpy()
                    if mod_type in hf:
                        dset = hf[mod_type]
                        cs = chunk_start // (embed_dim * 5 * 60)
                        ce = cs + data.shape[0]
                        if dset.shape[0] < ce:
                            dset.resize((ce,) + data.shape[1:])
                        dset[cs:ce] = data
                    else:
                        hf.create_dataset(mod_type, data=data,
                                          chunks=(embed_dim,) + data.shape[1:],
                                          maxshape=(None,) + data.shape[1:])

        # Save 5-second granular embeddings
        embeddings_5s = [e[1] for e in embeddings]
        for i in range(len(file_paths)):
            subject_id = os.path.basename(file_paths[i]).split('.')[0]
            chunk_start = chunk_starts[i]
            out_path = os.path.join(output_emb, f"{subject_id}.hdf5")
            with h5py.File(out_path, 'a') as hf:
                for mod_idx, mod_type in enumerate(modality_types):
                    data = embeddings_5s[mod_idx][i].cpu().numpy()
                    if mod_type in hf:
                        dset = hf[mod_type]
                        cs = chunk_start // (embed_dim * 5)
                        ce = cs + data.shape[0]
                        if dset.shape[0] < ce:
                            dset.resize((ce,) + data.shape[1:])
                        dset[cs:ce] = data
                    else:
                        hf.create_dataset(mod_type, data=data,
                                          chunks=(embed_dim,) + data.shape[1:],
                                          maxshape=(None,) + data.shape[1:])

print(f"\nEmbeddings saved!")
print(f"  5-second granular: {output_emb}")
print(f"  5-minute aggregated: {output_5min_agg}")

# Inspect output
with h5py.File(os.path.join(output_emb, "demo_psg.hdf5"), 'r') as f:
    for key in f.keys():
        print(f"  {key}: {f[key].shape}")

---
## 4. Sleep Staging Inference

The finetuned **SleepEventLSTMClassifier** (1.19M params) predicts 5 sleep stages:
- **Wake** (W)
- **Stage 1** (N1) — light sleep
- **Stage 2** (N2) — spindles & K-complexes
- **Stage 3** (N3) — deep/slow-wave sleep
- **REM** — rapid eye movement

Architecture: Spatial Attention → Transformer Encoder → Bidirectional LSTM → per-epoch classification

In [ ]:
# --- Helper classes for sleep staging (from demo notebook) ---

class SleepEventClassificationDataset(Dataset):
    def __init__(self, config, channel_groups, hdf5_paths, label_files, split="train"):
        self.config = config
        self.max_channels = self.config["max_channels"]
        self.context = int(self.config["context"])
        self.channel_like = self.config["channel_like"]
        self.max_seq_len = config["model_params"]["max_seq_length"]

        labels_dict = {
            os.path.basename(p).rsplit(".", 1)[0]: p
            for p in label_files if os.path.exists(p)
        }

        hdf5_paths = [p for p in hdf5_paths if os.path.exists(p)]
        hdf5_paths = [
            p for p in hdf5_paths
            if os.path.basename(p).rsplit(".", 1)[0] in labels_dict
        ]

        self.hdf5_paths = hdf5_paths
        self.labels_dict = labels_dict

        if self.context == -1:
            self.index_map = [
                (p, labels_dict[os.path.basename(p).rsplit(".", 1)[0]], -1)
                for p in self.hdf5_paths
            ]
        else:
            self.index_map = []
            for hdf5_file_path in self.hdf5_paths:
                file_prefix = os.path.basename(hdf5_file_path).rsplit(".", 1)[0]
                label_path = labels_dict[file_prefix]
                with h5py.File(hdf5_file_path, "r") as hf:
                    dset_names = list(hf.keys())
                    if len(dset_names) == 0:
                        continue
                    dataset_length = hf[dset_names[0]].shape[0]
                for i in range(0, dataset_length, self.context):
                    self.index_map.append((hdf5_file_path, label_path, i))

        self.total_len = len(self.index_map)

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        hdf5_path, label_path, start_index = self.index_map[idx]
        labels_df = pd.read_csv(label_path)
        labels_df["StageNumber"] = labels_df["StageNumber"].replace(-1, 0)
        y_data = labels_df["StageNumber"].to_numpy()
        if self.context != -1:
            y_data = y_data[start_index : start_index + self.context]

        x_data = []
        with h5py.File(hdf5_path, "r") as hf:
            for dataset_name in hf.keys():
                if dataset_name in self.channel_like:
                    if self.context == -1:
                        x_data.append(hf[dataset_name][:])
                    else:
                        x_data.append(hf[dataset_name][start_index : start_index + self.context])

        if not x_data:
            return self.__getitem__((idx + 1) % self.total_len)

        x_data = torch.tensor(np.array(x_data), dtype=torch.float32)
        y_data = torch.tensor(y_data, dtype=torch.float32)
        min_length = min(x_data.shape[1], len(y_data))
        x_data = x_data[:, :min_length, :]
        y_data = y_data[:min_length]

        return x_data, y_data, self.max_channels, self.max_seq_len, hdf5_path


def sleep_event_collate_fn(batch):
    x_data, y_data, max_channels_list, max_seq_len_list, hdf5_path_list = zip(*batch)
    num_channels = max(max_channels_list)
    max_seq_len_temp = max([item.size(1) for item in x_data])
    max_seq_len = min(max_seq_len_temp, max_seq_len_list[0]) if max_seq_len_list[0] else max_seq_len_temp

    padded_x_data, padded_y_data, padded_mask = [], [], []
    for x_item, y_item in zip(x_data, y_data):
        tgt = np.where(y_item.numpy() > 0, 1, 0)
        mavg = np.convolve(tgt, np.ones(1080)/1080, mode='valid')
        try:
            first_nz = np.where(mavg > 0.5)[0][0]
        except IndexError:
            first_nz = 0
        first_nz = max(first_nz, 0)

        c, s, e = x_item.size()
        c = min(c, num_channels)
        s = min(s, max_seq_len + first_nz)

        padded_x = torch.zeros((num_channels, max_seq_len, e))
        mask = torch.ones((num_channels, max_seq_len))
        padded_x[:c, :s-first_nz, :e] = x_item[:c, first_nz:s, :e]
        mask[:c, :s-first_nz] = 0

        padded_y = torch.zeros(max_seq_len)
        padded_y[:s-first_nz] = y_item[first_nz:s]

        padded_x_data.append(padded_x)
        padded_y_data.append(padded_y)
        padded_mask.append(mask)

    return torch.stack(padded_x_data), torch.stack(padded_y_data), torch.stack(padded_mask), hdf5_path_list

In [ ]:
# Load finetuned sleep staging model
ss_model_path = os.path.join(CHECKPOINTS, "model_sleep_staging")
ss_config = load_data(os.path.join(ss_model_path, "config.json"))

ss_model = SleepEventLSTMClassifier(**ss_config['model_params']).to(device)
ss_model = nn.DataParallel(ss_model)

ss_checkpoint = torch.load(os.path.join(ss_model_path, "best.pth"), map_location=device, weights_only=False)
ss_model.load_state_dict(ss_checkpoint)
ss_model.eval()

total_layers, total_params = count_parameters(ss_model)
print(f"Sleep staging model loaded: {total_params / 1e6:.2f}M params, {total_layers} layers")

In [ ]:
# Create dataset from generated embeddings
emb_paths = [os.path.join(DEMO_DATA, "demo_emb", "demo_psg.hdf5")]
label_files = [os.path.join(DEMO_DATA, "demo_psg.csv")]

test_dataset = SleepEventClassificationDataset(
    ss_config, channel_groups, split="test",
    hdf5_paths=emb_paths, label_files=label_files
)
test_loader = DataLoader(
    test_dataset, batch_size=8, shuffle=False,
    num_workers=1, collate_fn=sleep_event_collate_fn
)

print(f"Sleep staging dataset: {len(test_dataset)} samples")

In [ ]:
# Run sleep staging inference
all_targets, all_outputs, all_logits, all_masks, all_paths = [], [], [], [], []

with torch.no_grad():
    for (x_data, y_data, padded_matrix, hdf5_path_list) in tqdm.tqdm(test_loader, desc="Sleep staging"):
        x_data = x_data.to(device)
        y_data = y_data.to(device)
        padded_matrix = padded_matrix.to(device)

        outputs, mask = ss_model(x_data, padded_matrix)

        all_targets.append(y_data.cpu().numpy())
        all_outputs.append(torch.softmax(outputs, dim=-1).cpu().numpy())
        all_logits.append(outputs.cpu().numpy())
        all_masks.append(mask.cpu().numpy())
        all_paths.append(list(hdf5_path_list))

print(f"Predictions shape: {all_outputs[0].shape}")
print(f"  -> (batch, timesteps, 5_classes)")

In [ ]:
# Flatten and filter by mask
all_logits_flat = np.concatenate([l.reshape(-1, l.shape[-1]) for l in all_logits], axis=0)
all_outputs_flat = np.concatenate([o.reshape(-1, o.shape[-1]) for o in all_outputs], axis=0)
all_targets_flat = np.concatenate([t.reshape(-1) for t in all_targets], axis=0)
all_masks_flat = np.concatenate([m.reshape(-1) for m in all_masks], axis=0)

# mask==0 means real data (not padding)
mask_filter = all_masks_flat == 0
targets_filtered = all_targets_flat[mask_filter]
outputs_filtered = all_outputs_flat[mask_filter]
predicted_labels = np.argmax(outputs_filtered, axis=1)

print(f"Total epochs: {len(targets_filtered)}")
counts = Counter(targets_filtered)
class_labels = ["Wake", "Stage 1", "Stage 2", "Stage 3", "REM"]
for i, label in enumerate(class_labels):
    pct = counts.get(float(i), 0) / len(targets_filtered) * 100
    print(f"  {label}: {counts.get(float(i), 0)} ({pct:.1f}%)")

### 4a. Confusion Matrix

In [ ]:
# Per-class F1 scores
f1_scores = f1_score(targets_filtered, predicted_labels, average=None, labels=range(5))
print("Per-class F1 Scores:")
for idx, label in enumerate(class_labels):
    print(f"  {label}: {f1_scores[idx]:.3f}")
print(f"  Macro F1: {np.mean(f1_scores):.3f}")

# Normalized confusion matrix
conf_matrix = confusion_matrix(targets_filtered, predicted_labels, labels=range(5))
conf_matrix_pct = conf_matrix / conf_matrix.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    conf_matrix_pct, annot=True, fmt=".1f", cmap="Blues",
    xticklabels=class_labels, yticklabels=class_labels,
    annot_kws={"size": 13}, cbar_kws={"shrink": 0.8, "label": "% of true class"},
    ax=ax
)
ax.set_xlabel("Predicted Label", fontsize=13)
ax.set_ylabel("True Label", fontsize=13)
ax.set_title("Sleep Staging — Normalized Confusion Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4b. Hypnogram (Sleep Stage Timeline)

In [ ]:
# Hypnogram: predicted vs true sleep stages over time
# Each epoch = 5 seconds (granular embeddings at 128 Hz, 640-sample patches)
epoch_duration_sec = 5
n_epochs = len(targets_filtered)
time_hours = np.arange(n_epochs) * epoch_duration_sec / 3600

# Stage order for hypnogram (Wake at top, N3 at bottom, REM between)
# Standard hypnogram order: W, REM, N1, N2, N3 (top to bottom)
stage_remap = {0: 0, 4: 1, 1: 2, 2: 3, 3: 4}  # original -> display position
display_labels = ["Wake", "REM", "N1", "N2", "N3"]

true_display = np.array([stage_remap[int(s)] for s in targets_filtered])
pred_display = np.array([stage_remap[int(s)] for s in predicted_labels])

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

colors_true = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']

for ax, data, title in zip(axes, [true_display, pred_display], ['True (Ground Truth)', 'Predicted (SleepFM)']):
    ax.step(time_hours, data, where='post', linewidth=0.8, color='#333333')
    ax.fill_between(time_hours, data, step='post', alpha=0.3, color='#2196F3')
    ax.set_yticks(range(5))
    ax.set_yticklabels(display_labels)
    ax.set_ylim(-0.5, 4.5)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

axes[-1].set_xlabel('Time (hours)', fontsize=12)
fig.suptitle('Sleep Hypnogram', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4c. Prediction Confidence Over Time

In [ ]:
# Show model confidence (max softmax probability) over the night
confidence = np.max(outputs_filtered, axis=1)

fig, ax = plt.subplots(figsize=(16, 3))
ax.plot(time_hours, confidence, linewidth=0.5, alpha=0.6, color='#2196F3')

# Rolling average for trend
window = min(120, len(confidence) // 4)  # ~10 min window
if window > 1:
    rolling_conf = np.convolve(confidence, np.ones(window)/window, mode='valid')
    rolling_time = time_hours[:len(rolling_conf)]
    ax.plot(rolling_time, rolling_conf, linewidth=2, color='#F44336', label=f'{window*5/60:.0f}-min rolling avg')

ax.set_xlabel('Time (hours)', fontsize=12)
ax.set_ylabel('Confidence', fontsize=12)
ax.set_title('Prediction Confidence Over Time', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Disease Prediction Inference

The finetuned **DiagnosisFinetuneFullLSTMCOXPHWithDemo** model (0.91M params) outputs
Cox proportional hazard log-probabilities for **1,065 medical conditions**.

Architecture: Spatial Attention → Bidirectional LSTM → Mean Pooling + Demographics → FC

Input:
- 5-second granular embeddings (4 modalities)
- Demographics: age + gender

In [ ]:
# --- Helper classes for disease prediction ---

class DiagnosisFinetuneFullCOXPHWithDemoDataset(Dataset):
    def __init__(self, config, channel_groups, hdf5_paths=None,
                 demo_labels_path=None, split="train"):
        self.config = config
        self.channel_groups = channel_groups
        self.max_channels = self.config["max_channels"]

        if not demo_labels_path:
            demo_labels_path = config["demo_labels_path"]

        demo_labels_df = pd.read_csv(demo_labels_path).set_index("Study ID")
        study_ids = set(demo_labels_df.index)

        is_event_df = pd.read_csv(os.path.join(self.config["labels_path"], "is_event.csv")).set_index('Study ID')
        event_time_df = pd.read_csv(os.path.join(self.config["labels_path"], "time_to_event.csv")).set_index('Study ID')

        if hdf5_paths:
            hdf5_paths = [f for f in hdf5_paths if os.path.exists(f)]
        else:
            split_paths = load_data(config["split_path"])[split]
            hdf5_paths = [f for f in split_paths if os.path.exists(f)]

        hdf5_paths = [f for f in hdf5_paths if os.path.basename(f).split(".")[0] in study_ids]

        labels_dict = {}
        for study_id in study_ids:
            labels_dict[study_id] = {
                "is_event": list(is_event_df.loc[study_id].values),
                "event_time": list(event_time_df.loc[study_id].values),
                "demo_feats": list(demo_labels_df.loc[study_id].values)
            }

        self.index_map = [
            (path, labels_dict[os.path.basename(path).split(".")[0]])
            for path in hdf5_paths
        ]
        self.total_len = len(self.index_map)
        self.max_seq_len = config["model_params"]["max_seq_length"]
        print(f"Disease prediction dataset: {self.total_len} subjects")

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        hdf5_path, tte_event = self.index_map[idx]
        event_time = torch.tensor(tte_event["event_time"], dtype=torch.float32)
        is_event = torch.tensor(tte_event["is_event"])
        demo_feats = torch.tensor(tte_event["demo_feats"], dtype=torch.float32)

        x_data = []
        with h5py.File(hdf5_path, 'r') as hf:
            dset_names = [k for k in hf.keys() if k in self.config["modality_types"]]
            random.shuffle(dset_names)
            for name in dset_names:
                x_data.append(hf[name][:])

        if not x_data:
            return self.__getitem__((idx + 1) % self.total_len)

        x_data = torch.tensor(np.array(x_data), dtype=torch.float32)
        return x_data, event_time, is_event, demo_feats, self.max_channels, self.max_seq_len, hdf5_path


def diagnosis_collate_fn(batch):
    x_data, event_time, is_event, demo_feats, max_ch_list, max_seq_list, paths = zip(*batch)
    num_channels = max(max_ch_list)
    max_seq_len = max_seq_list[0] if max_seq_list[0] else max([item.size(1) for item in x_data])

    padded_x, padded_mask = [], []
    for item in x_data:
        c, s, e = item.size()
        c = min(c, num_channels)
        s = min(s, max_seq_len)
        px = torch.zeros((num_channels, max_seq_len, e))
        mk = torch.ones((num_channels, max_seq_len))
        px[:c, :s, :e] = item[:c, :s, :e]
        mk[:c, :s] = 0
        padded_x.append(px)
        padded_mask.append(mk)

    return (torch.stack(padded_x), torch.stack(event_time), torch.stack(is_event),
            torch.stack(demo_feats), torch.stack(padded_mask), paths)

In [ ]:
# Load finetuned disease prediction model
dx_model_path = os.path.join(CHECKPOINTS, "model_diagnosis")
dx_config = load_data(os.path.join(dx_model_path, "config.json"))
dx_config["model_params"]["dropout"] = 0.0  # No dropout for inference

dx_model = DiagnosisFinetuneFullLSTMCOXPHWithDemo(**dx_config['model_params']).to(device)
dx_model = nn.DataParallel(dx_model)

dx_checkpoint = torch.load(os.path.join(dx_model_path, "best.pth"), map_location=device, weights_only=False)
dx_model.load_state_dict(dx_checkpoint)
dx_model.eval()

total_layers, total_params = count_parameters(dx_model)
print(f"Disease prediction model loaded: {total_params / 1e6:.2f}M params, {total_layers} layers")

In [ ]:
# Create disease prediction dataset
dx_config["labels_path"] = DEMO_DATA  # where is_event.csv and time_to_event.csv are
emb_paths = [os.path.join(DEMO_DATA, "demo_emb", "demo_psg.hdf5")]
demo_labels_path = os.path.join(DEMO_DATA, "demo_age_gender.csv")

dx_dataset = DiagnosisFinetuneFullCOXPHWithDemoDataset(
    dx_config, channel_groups, split="test",
    hdf5_paths=emb_paths, demo_labels_path=demo_labels_path
)
dx_loader = DataLoader(
    dx_dataset, batch_size=8, shuffle=False,
    num_workers=1, collate_fn=diagnosis_collate_fn
)

In [ ]:
# Run disease prediction inference
all_dx_outputs, all_event_times, all_is_event, all_dx_paths = [], [], [], []

with torch.no_grad():
    for item in tqdm.tqdm(dx_loader, desc="Disease prediction"):
        x_data, event_times, is_event, demo_feats, padded_matrix, paths = item
        x_data = x_data.to(device)
        event_times = event_times.to(device)
        is_event = is_event.to(device)
        demo_feats = demo_feats.to(device)
        padded_matrix = padded_matrix.to(device)

        outputs = dx_model(x_data, padded_matrix, demo_feats)

        all_dx_outputs.append(outputs.cpu().numpy())
        all_event_times.append(event_times.cpu().numpy())
        all_is_event.append(is_event.cpu().numpy())
        all_dx_paths.append(list(paths))

all_dx_outputs = np.concatenate(all_dx_outputs, axis=0)
all_event_times = np.concatenate(all_event_times, axis=0)
all_is_event = np.concatenate(all_is_event, axis=0)

print(f"\nOutput shape: {all_dx_outputs.shape}")
print(f"  -> ({all_dx_outputs.shape[0]} subjects, {all_dx_outputs.shape[1]} conditions)")

In [ ]:
# Map outputs to disease labels
labels_df = pd.read_csv(os.path.join(CONFIGS, "label_mapping.csv"))
labels_df["output"] = all_dx_outputs[0]
labels_df["is_event"] = all_is_event[0]
labels_df["event_time"] = all_event_times[0]

print(f"Disease vocabulary: {len(labels_df)} conditions")
print(f"\nSample predictions (first 10):")
labels_df[["phenotype", "phecode", "output", "is_event"]].head(10)

### 5a. Top Disease Risk Predictions

In [ ]:
# Top 30 predicted conditions ranked by hazard score
top_n = 30
top_diseases = labels_df.nlargest(top_n, 'output')[['phenotype', 'phecode', 'output', 'is_event']]

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#F44336' if ev else '#2196F3' for ev in top_diseases['is_event'].values]

bars = ax.barh(
    range(top_n), top_diseases['output'].values,
    color=colors, alpha=0.8, edgecolor='white'
)
ax.set_yticks(range(top_n))
ax.set_yticklabels(
    [f"{row.phenotype} ({row.phecode})" for _, row in top_diseases.iterrows()],
    fontsize=9
)
ax.invert_yaxis()
ax.set_xlabel('Hazard Log-Probability', fontsize=12)
ax.set_title(f'Top {top_n} Predicted Disease Risks', fontsize=14, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#F44336', alpha=0.8, label='Event occurred (is_event=1)'),
    Patch(facecolor='#2196F3', alpha=0.8, label='No event (is_event=0)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTop 10 highest-risk conditions:")
for i, (_, row) in enumerate(top_diseases.head(10).iterrows()):
    event_marker = " [EVENT]" if row.is_event else ""
    print(f"  {i+1}. {row.phenotype} (PheCode {row.phecode}) — score: {row.output:.2f}{event_marker}")

### 5b. Disease Category Distribution

In [ ]:
# Group diseases by PheCode category (first digit = organ system)
def phecode_category(code):
    try:
        first = int(float(code))
    except (ValueError, TypeError):
        return "Other"
    categories = {
        range(1, 140): "Infectious",
        range(140, 240): "Neoplasms",
        range(240, 280): "Endocrine/Metabolic",
        range(280, 290): "Blood/Immune",
        range(290, 320): "Mental/Behavioral",
        range(320, 390): "Nervous System",
        range(390, 460): "Circulatory",
        range(460, 520): "Respiratory",
        range(520, 580): "Digestive",
        range(580, 630): "Genitourinary",
        range(680, 710): "Skin",
        range(710, 740): "Musculoskeletal",
        range(740, 760): "Congenital",
        range(780, 800): "Symptoms",
        range(800, 1000): "Injury/External",
    }
    for r, cat in categories.items():
        if first in r:
            return cat
    return "Other"

labels_df['category'] = labels_df['phecode'].apply(phecode_category)

# Average hazard score per category
cat_scores = labels_df.groupby('category')['output'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
cat_scores.plot.barh(ax=ax, color='#4CAF50', alpha=0.8, edgecolor='white')
ax.set_xlabel('Mean Hazard Log-Probability', fontsize=12)
ax.set_title('Average Disease Risk by Category', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Advanced Visualizations

### 6a. Embedding t-SNE

Visualize how the learned embeddings cluster across the 4 PSG modalities.

In [ ]:
# Load 5-minute aggregated embeddings for t-SNE
emb_file = os.path.join(DEMO_DATA, "demo_5min_agg_emb", "demo_psg.hdf5")

modality_embeddings = {}
with h5py.File(emb_file, 'r') as f:
    for key in f.keys():
        modality_embeddings[key] = f[key][:]
        print(f"  {key}: {f[key].shape}")

# Concatenate all modality embeddings with labels
all_embs = []
all_labels = []
for mod_name, emb in modality_embeddings.items():
    # Flatten if needed: (N, 1, D) -> (N, D)
    if emb.ndim == 3:
        emb = emb.reshape(emb.shape[0], -1)
    all_embs.append(emb)
    all_labels.extend([mod_name] * len(emb))

all_embs = np.concatenate(all_embs, axis=0)
print(f"\nTotal embeddings for t-SNE: {all_embs.shape}")

In [ ]:
# Run t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_embs)-1))
embs_2d = tsne.fit_transform(all_embs)

# Plot
mod_colors = {'BAS': '#2196F3', 'RESP': '#4CAF50', 'EKG': '#F44336', 'EMG': '#FF9800'}
mod_names = {'BAS': 'Brain/Sleep (EEG+EOG)', 'RESP': 'Respiratory', 'EKG': 'Cardiac (ECG)', 'EMG': 'Muscle (EMG)'}

fig, ax = plt.subplots(figsize=(10, 8))
for mod in modality_types:
    mask = np.array(all_labels) == mod
    ax.scatter(
        embs_2d[mask, 0], embs_2d[mask, 1],
        c=mod_colors.get(mod, '#999999'), label=mod_names.get(mod, mod),
        alpha=0.7, s=60, edgecolors='white', linewidth=0.5
    )

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('SleepFM Embedding Space — t-SNE by Modality', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 6b. Embedding t-SNE Colored by Sleep Stage

In [ ]:
# Load 5-second embeddings for sleep-stage colored t-SNE
emb_5s_file = os.path.join(DEMO_DATA, "demo_emb", "demo_psg.hdf5")
labels_csv = pd.read_csv(os.path.join(DEMO_DATA, "demo_psg.csv"))
stage_labels = labels_csv["StageNumber"].replace(-1, 0).values

with h5py.File(emb_5s_file, 'r') as f:
    # Use BAS modality (brain activity — most relevant to sleep stages)
    bas_emb = f['BAS'][:]
    print(f"BAS embeddings: {bas_emb.shape}")
    print(f"Stage labels: {stage_labels.shape}")

# Align lengths
n = min(len(bas_emb), len(stage_labels))
bas_emb = bas_emb[:n]
stage_labels_aligned = stage_labels[:n]

# Subsample if too many points (t-SNE is slow for >5000)
max_points = 3000
if n > max_points:
    idx = np.random.choice(n, max_points, replace=False)
    idx.sort()
    bas_emb = bas_emb[idx]
    stage_labels_aligned = stage_labels_aligned[idx]

# t-SNE
tsne2 = TSNE(n_components=2, random_state=42, perplexity=min(30, len(bas_emb)-1))
embs_2d_stage = tsne2.fit_transform(bas_emb)

# Plot colored by sleep stage
stage_colors = {0: '#FFD54F', 1: '#81C784', 2: '#4FC3F7', 3: '#7E57C2', 4: '#FF8A65'}
stage_names = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}

fig, ax = plt.subplots(figsize=(10, 8))
for stage in sorted(stage_colors.keys()):
    mask = stage_labels_aligned == stage
    if mask.sum() > 0:
        ax.scatter(
            embs_2d_stage[mask, 0], embs_2d_stage[mask, 1],
            c=stage_colors[stage], label=stage_names[stage],
            alpha=0.6, s=40, edgecolors='white', linewidth=0.3
        )

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('BAS Embedding Space — Colored by Sleep Stage', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, markerscale=1.5, title='Sleep Stage')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---
## 7. Using Real Data (Stanford Sleep Dataset)

The synthetic demo data above proves the pipeline works. To run on real PSG recordings:

### Step 1: Access the Data

The Stanford Sleep Cohort (SSC) dataset is available at:  
**https://bdsp.io/content/08vg8vqv2wdtwonc1ddy/1.0**

Requirements:
- Free account on bdsp.io
- Signed data use agreement (PhysioNet-style)
- Download 5-10 EDF files to start

### Step 2: Upload to Google Drive

Upload your EDF files to `My Drive/sleepfm-data/edf/`

### Step 3: Preprocess

```python
# Batch preprocess all EDFs in a directory
converter = EDFToHDF5Converter(
    root_dir='/content/drive/MyDrive/sleepfm-data/edf/',
    target_dir='/content/drive/MyDrive/sleepfm-data/hdf5/',
    resample_rate=128
)
converter.convert_all_multiprocessing()
```

### Step 4: Generate Embeddings & Run Inference

Follow the same pipeline as above (Sections 3-5) but point to your HDF5 files.

### Important Notes

- **Channel mapping**: Review your EDF channels against `sleepfm/configs/channel_groups.json`. Different PSG systems use different channel names — update the mapping if needed.
- **Finetuning recommended**: Even with a handful of labeled samples, finetuning the sleep staging head on your specific data distribution will improve results. Use `sleepfm/pipeline/finetune_sleep_staging.py`.
- **Disease labels**: For disease prediction evaluation, you need `is_event.csv`, `time_to_event.csv`, and `demo_age_gender.csv` matching your cohort.
- **GPU memory**: T4 (16GB) is sufficient for inference. A100 recommended for finetuning.

In [ ]:
# Template for real data (uncomment and modify paths)

# REAL_DATA_DIR = '/content/drive/MyDrive/sleepfm-data'
# EDF_DIR = os.path.join(REAL_DATA_DIR, 'edf')
# HDF5_DIR = os.path.join(REAL_DATA_DIR, 'hdf5')
# EMB_DIR = os.path.join(REAL_DATA_DIR, 'embeddings')

# # Step 1: Preprocess
# converter = EDFToHDF5Converter(
#     root_dir=EDF_DIR,
#     target_dir=HDF5_DIR,
#     resample_rate=128
# )
# converter.convert_all_multiprocessing()

# # Step 2: Generate embeddings
# import glob
# hdf5_files = glob.glob(os.path.join(HDF5_DIR, '*.hdf5'))
# print(f'Found {len(hdf5_files)} HDF5 files')

# dataset = SetTransformerDataset(config, channel_groups, hdf5_paths=hdf5_files, split='test')
# dataloader = DataLoader(dataset, batch_size=16, num_workers=2, shuffle=False, collate_fn=collate_fn)
# # ... (run embedding generation loop from Section 3)

# # Step 3: Sleep staging (need label files for evaluation)
# # ... (follow Section 4 with your embedding paths)

# # Step 4: Disease prediction (need demographics + event labels)
# # ... (follow Section 5 with your embedding paths)

---

## Summary

This notebook demonstrated the full SleepFM-Clinical inference pipeline:

| Step | Model | Params | Output |
|------|-------|--------|--------|
| Preprocessing | EDFToHDF5Converter | — | Raw signals → HDF5 |
| Embedding | SetTransformer | 4.44M | 128-dim embeddings per modality |
| Sleep Staging | SleepEventLSTMClassifier | 1.19M | 5-class per-epoch predictions |
| Disease Prediction | DiagnosisCoxPH+Demo | 0.91M | 1,065-condition risk scores |

### Key Takeaways

1. **Multimodal**: The model processes 4 PSG modalities (BAS, RESP, EKG, EMG) independently, then fuses via attention — each modality contributes unique signal.

2. **Foundation model**: The contrastive pretrained SetTransformer learns general-purpose sleep representations that transfer to multiple downstream tasks.

3. **Clinical utility**: Disease prediction from a single overnight PSG could enable early screening for conditions like dementia, cardiovascular disease, and metabolic disorders.

### References

- Paper: [SleepFM: Multi-modal Foundation Model for Sleep](https://www.nature.com/articles/s41591-024-03392-9) (Nature Medicine, 2026)
- Code: [github.com/zou-group/sleepfm-clinical](https://github.com/zou-group/sleepfm-clinical)
- Data: [Stanford Sleep Cohort on BDSP](https://bdsp.io/content/08vg8vqv2wdtwonc1ddy/1.0)